# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayesha-asif1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## My rule and its reason codes

### The rule (plain words)

For every page, I compare its actual CTR to the "expected CTR" for pages at the same search position (top_3, top_10, top_20, beyond_20). The gap between expected and actual CTR is the raw opportunity signal — a bigger gap means the page is underperforming more than similar pages.

But a raw gap alone can be misleading: a page with only 1-2 impressions and zero clicks looks identical to a page with 500 impressions and zero clicks, even though the second case is a much stronger, more trustworthy signal. So I weight the opportunity gap by log(1 + impressions) — pages with more impressions get more confidence in their score, while very-low-impression pages are naturally pulled down the ranking instead of dominating the top of the list on a fluke.

**Formula:**
- expected_ctr = average CTR of all pages in the same position_bucket
- opportunity_gap = expected_ctr − actual_ctr
- confidence_weighted_score = opportunity_gap × log(1 + gsc_impressions)

Pages are ranked by confidence_weighted_score, descending. A higher score means: this page is underperforming its position AND has enough traffic volume to trust that the underperformance is real, not noise.

### Reason codes

Each flagged page gets one of these labels, based on its position_bucket and impression volume:

| Reason code | Condition | What it means |
|---|---|---|
| `high_position_low_ctr` | position_bucket in (top_3, top_10) AND opportunity_gap > 0 | Page ranks well but is losing clicks it should be getting — likely a title/meta-description fix |
| `mid_position_low_ctr` | position_bucket == top_20 AND opportunity_gap > 0 | Decent ranking, underperforming — worth a content/metadata review |
| `low_confidence_signal` | gsc_impressions < 10 | Too few impressions to trust the CTR number — flag for monitoring only, not immediate action |
| `on_par_or_above` | opportunity_gap <= 0 | Page is meeting or beating its position's expected CTR — no action needed |

In [1]:
# --- Load data (same as w03_data_contract) ---
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=hf_token)

import pandas as pd
import numpy as np

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

# --- Filter to reliable rows ---
df_available = df_march[df_march['gsc_data_available'] == True].copy()

# --- Label ---
df_available['ctr'] = df_available['gsc_clicks'] / df_available['gsc_impressions']

# --- Features ---
df_available['day_of_week'] = pd.to_datetime(df_available['report_date']).dt.dayofweek
df_available['log_impressions'] = np.log1p(df_available['gsc_impressions'])

def position_bucket(pos):
    if pos <= 3:
        return 'top_3'
    elif pos <= 10:
        return 'top_10'
    elif pos <= 20:
        return 'top_20'
    else:
        return 'beyond_20'

df_available['position_bucket'] = df_available['gsc_avg_position'].apply(position_bucket)

# --- Final feature vector ---
feature_frame = df_available[[
    'client_hash_id', 'content_hash_id', 'report_date',   # context
    'ctr',                                                  # label
    'gsc_impressions', 'gsc_avg_position', 'day_of_week',   # raw features
    'log_impressions', 'position_bucket'                    # engineered features
]].copy()

print("Feature vector shape:", feature_frame.shape)
feature_frame.head()


Feature vector shape: (3611061, 9)


,client_hash_id,content_hash_id,report_date,ctr,gsc_impressions,gsc_avg_position,day_of_week,log_impressions,position_bucket
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0.000,20,3.350000,6,3.044522,top_10
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0.000,1,0.000000,6,0.693147,top_3
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,0.008,125,4.928000,6,4.836282,top_10
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0.000,7,4.000000,6,2.079442,top_10
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0.000,11,2.272727,6,2.484907,top_3


In [3]:
expected_ctr_by_bucket = feature_frame.groupby('position_bucket')['ctr'].mean()
print(expected_ctr_by_bucket)

position_bucket
beyond_20    0.001289
top_10       0.003473
top_20       0.002770
top_3        0.004756
Name: ctr, dtype: float64


In [4]:
# Map each row's position_bucket to its expected CTR
feature_frame['expected_ctr'] = feature_frame['position_bucket'].map(expected_ctr_by_bucket)

# Opportunity gap: kitna CTR "chhoot" raha hai expected se
feature_frame['opportunity_gap'] = feature_frame['expected_ctr'] - feature_frame['ctr']

# Top opportunities dekho — sabse zyada gap wale pages
top_opportunities = feature_frame.sort_values('opportunity_gap', ascending=False).head(10)
print(top_opportunities[['content_hash_id', 'position_bucket', 'ctr', 'expected_ctr', 'opportunity_gap']])

                  content_hash_id position_bucket  ctr  expected_ctr  \
1        content_05597932fe4da067           top_3  0.0      0.004756   
9841356  content_e59204eb82d02e13           top_3  0.0      0.004756   
4        content_a3ea9792f793ec72           top_3  0.0      0.004756   
9841350  content_c5d5282bba9a3f18           top_3  0.0      0.004756   
9841370  content_127f40ef6b42cec2           top_3  0.0      0.004756   
5214544  content_9330df9595681094           top_3  0.0      0.004756   
5214546  content_15dfdbe6773325c3           top_3  0.0      0.004756   
5214550  content_46fd3db239faab68           top_3  0.0      0.004756   
5214520  content_66f58dca79136ad4           top_3  0.0      0.004756   
5214521  content_716e136f4613c25b           top_3  0.0      0.004756   

         opportunity_gap  
1               0.004756  
9841356         0.004756  
4               0.004756  
9841350         0.004756  
9841370         0.004756  
5214544         0.004756  
5214546         0.

In [5]:
print(top_opportunities[['content_hash_id', 'position_bucket', 'gsc_impressions', 'ctr', 'expected_ctr', 'opportunity_gap']])

                  content_hash_id position_bucket  gsc_impressions  ctr  \
1        content_05597932fe4da067           top_3                1  0.0   
9841356  content_e59204eb82d02e13           top_3               23  0.0   
4        content_a3ea9792f793ec72           top_3               11  0.0   
9841350  content_c5d5282bba9a3f18           top_3                6  0.0   
9841370  content_127f40ef6b42cec2           top_3               96  0.0   
5214544  content_9330df9595681094           top_3               79  0.0   
5214546  content_15dfdbe6773325c3           top_3               31  0.0   
5214550  content_46fd3db239faab68           top_3                3  0.0   
5214520  content_66f58dca79136ad4           top_3                5  0.0   
5214521  content_716e136f4613c25b           top_3               37  0.0   

         expected_ctr  opportunity_gap  
1            0.004756         0.004756  
9841356      0.004756         0.004756  
4            0.004756         0.004756  
9841350   

In [6]:
def assign_reason_code(row):
    if row['gsc_impressions'] < 10:
        return 'low_confidence_signal'
    elif row['opportunity_gap'] <= 0:
        return 'on_par_or_above'
    elif row['position_bucket'] in ('top_3', 'top_10'):
        return 'high_position_low_ctr'
    elif row['position_bucket'] == 'top_20':
        return 'mid_position_low_ctr'
    else:
        return 'monitor_only'  # beyond_20 with a gap — low priority, position itself is the limiter

# Quick test on a small sample before applying to full data
sample = feature_frame.head(10).copy()
sample['expected_ctr'] = sample['position_bucket'].map(expected_ctr_by_bucket)
sample['opportunity_gap'] = sample['expected_ctr'] - sample['ctr']
sample['reason_code'] = sample.apply(assign_reason_code, axis=1)

print(sample[['content_hash_id', 'position_bucket', 'gsc_impressions', 'ctr', 'opportunity_gap', 'reason_code']])

            content_hash_id position_bucket  gsc_impressions       ctr  \
0  content_b7e512995f79d5a6          top_10               20  0.000000   
1  content_05597932fe4da067           top_3                1  0.000000   
2  content_7a105f548d9c6916          top_10              125  0.008000   
3  content_905aa32a0230694e          top_10                7  0.000000   
4  content_a3ea9792f793ec72           top_3               11  0.000000   
5  content_36c36abc7650d7af          top_10              239  0.004184   
6  content_a7da352b73b02668          top_10              191  0.000000   
7  content_05434271b257bb68          top_10               55  0.000000   
8  content_d056587ff7faca0c          top_10               77  0.000000   
9  content_bfd1e41c2af250c8          top_10                2  0.000000   

   opportunity_gap            reason_code  
0         0.003473  high_position_low_ctr  
1         0.004756  low_confidence_signal  
2        -0.004527        on_par_or_above  
3        

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import numpy as np

# Expected CTR per position bucket (already computed earlier)
feature_frame['expected_ctr'] = feature_frame['position_bucket'].map(expected_ctr_by_bucket)

# Opportunity gap
feature_frame['opportunity_gap'] = feature_frame['expected_ctr'] - feature_frame['ctr']

# Confidence-weighted score
feature_frame['confidence_weighted_score'] = feature_frame['opportunity_gap'] * np.log1p(feature_frame['gsc_impressions'])

# Reason code for every row
feature_frame['reason_code'] = feature_frame.apply(assign_reason_code, axis=1)

# Rank: highest score = biggest, most trustworthy opportunity
ranked_queue = feature_frame.sort_values('confidence_weighted_score', ascending=False).reset_index(drop=True)

print("Total ranked rows:", len(ranked_queue))
print(ranked_queue[['content_hash_id', 'position_bucket', 'gsc_impressions', 'ctr',
                     'opportunity_gap', 'confidence_weighted_score', 'reason_code']].head(15))

Total ranked rows: 3611061
             content_hash_id position_bucket  gsc_impressions       ctr  \
0   content_44f34c0a90047651           top_3            40084  0.000025   
1   content_34a70fea29d15f24           top_3            39003  0.000051   
2   content_fec55986a1868d62           top_3            33383  0.000000   
3   content_44f34c0a90047651           top_3            32958  0.000000   
4   content_fec55986a1868d62           top_3            31472  0.000000   
5   content_9c057b66c30a3abb           top_3            28973  0.000000   
6   content_9c057b66c30a3abb           top_3            28947  0.000000   
7   content_44f34c0a90047651           top_3            30964  0.000032   
8   content_44f34c0a90047651           top_3            32756  0.000061   
9   content_44f34c0a90047651           top_3            30791  0.000065   
10  content_44f34c0a90047651           top_3            30573  0.000065   
11  content_9c057b66c30a3abb           top_3            24233  0.000000  

In [11]:
import os
os.makedirs('work/outputs', exist_ok=True)

output_cols = ['client_hash_id', 'content_hash_id', 'position_bucket',
               'total_impressions', 'total_clicks', 'avg_position', 'ctr',
               'expected_ctr', 'opportunity_gap', 'confidence_weighted_score', 'reason_code']

ranked_pages[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Saved:", len(ranked_pages), "rows to work/outputs/baseline_action_score.csv")

Saved: 176738 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

## Top-20 review

**Pattern observed:** All 20 top-ranked pages are in the top_3 position bucket with the reason code high_position_low_ctr. This makes sense given how the score is built — top_3 has the highest expected CTR (1.24%), so pages that fail to meet it produce the largest opportunity_gap, and all 20 also have large impression volumes (23K–203K), so the confidence weighting doesn't discount them. This means the current baseline surfaces the highest-stakes cases first: pages winning the ranking battle but losing the click battle.

| # | Page | Impressions | CTR vs Expected | Action | Confidence note | What would make this wrong |
|---|---|---|---|---|---|---|
| 1 | content_306bc78d... | 80,821 | 0.04% vs 1.24% | Rewrite title/meta description | Very high — large sample size | Recent publish still building trust, or purely informational query |
| 2 | content_8d7d99f1... | 203,497 | 0.14% vs 1.24% | Rewrite title/meta description | Very high — largest sample in top 20 | Query may be a definitional/quick-answer type where users don't need to click |
| 3 | content_fc677590... | 60,172 | 0.03% vs 1.24% | Rewrite title/meta description | Very high | Snippet may already be optimal if intent is purely navigational |
| 4 | content_c46df0fa... | 70,398 | 0.06% vs 1.24% | Rewrite title/meta description | Very high | Featured snippet or "position zero" could be absorbing clicks before this result |
| 5 | content_d61fc394... | 38,000 | 0.003% vs 1.24% (near-zero) | Urgent metadata review | Very high — near-zero CTR despite large volume is a strong signal | Possible technical issue (broken link, wrong title rendering) rather than just weak copy |
| 6 | content_9ef3d751... | 89,229 | 0.10% vs 1.24% | Rewrite title/meta description | Very high | Could be a branded query where users already know the content, reducing click need |
| 7 | content_b2b85c28... | 65,304 | 0.09% vs 1.24% | Rewrite title/meta description | Very high | Same as above — brand recognition or snippet-sufficient answers |
| 8 | content_7f52754c... | 43,135 | 0.06% vs 1.24% | Rewrite title/meta description | High | Seasonal/topical query where interest recently dropped, unrelated to metadata |
| 9 | content_252aa548... | 66,698 | 0.11% vs 1.24% | Rewrite title/meta description | Very high | Could rank for a broad/ambiguous query where many results look similar |
| 10 | content_b9acd1eb... | 25,941 | 0.01% vs 1.24% | Rewrite title/meta description | High | Smaller sample than top rows — still solid, but slightly less certain |
| 11 | content_66bf45eb... | 24,259 | 0.004% vs 1.24% | Urgent metadata review | High — near-zero CTR | Possible duplicate/thin content competing with a stronger page from the same site |
| 12 | content_805fd455... | 30,792 | 0.04% vs 1.24% | Rewrite title/meta description | High | Could be cannibalized by another page on the same site ranking nearby |
| 13 | content_a07d1e32... | 35,083 | 0.06% vs 1.24% | Rewrite title/meta description | High | Same cannibalization risk |
| 14 | content_1d7764b6... | 23,402 | 0.02% vs 1.24% | Rewrite title/meta description | High | Smaller sample — still reliable, but treat as slightly lower certainty |
| 15 | content_ff894194... | 44,217 | 0.09% vs 1.24% | Rewrite title/meta description | High | — |
| 16 | content_020dd576... | 58,588 | 0.12% vs 1.24% | Rewrite title/meta description | High | — |
| 17 | content_04fb6296... | 28,737 | 0.05% vs 1.24% | Rewrite title/meta description | High | — |
| 18 | content_97d7732b... | 29,101 | 0.05% vs 1.24% | Rewrite title/meta description | High | — |
| 19 | content_8028c500... | 27,747 | 0.05% vs 1.24% | Rewrite title/meta description | High | — |
| 20 | content_0d1d0886... | 51,722 | 0.12% vs 1.24% | Rewrite title/meta description | High | — |

In [9]:
# Step 1: aggregate raw counts per unique page (across all days in March)
page_level = df_available.groupby(['client_hash_id', 'content_hash_id']).agg(
    total_impressions=('gsc_impressions', 'sum'),
    total_clicks=('gsc_clicks', 'sum'),
    avg_position=('gsc_avg_position', 'mean')   # position ke liye average theek hai
).reset_index()

# Step 2: recompute CTR correctly (total clicks / total impressions, not average of daily CTRs)
page_level['ctr'] = page_level['total_clicks'] / page_level['total_impressions']

# Step 3: rebuild position_bucket on the page-level avg_position
page_level['position_bucket'] = page_level['avg_position'].apply(position_bucket)

print("Unique pages:", len(page_level))
page_level.head()

Unique pages: 176738


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,ctr,position_bucket
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.000000,0.000000,top_10
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,14.129210,0.006042,top_20
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,9.225529,0.000000,top_10
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,8.470926,0.000000,top_10
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,14.859827,0.000000,top_20


In [10]:
# Step 1: recompute expected CTR per bucket, on page-level data
expected_ctr_by_bucket_page = page_level.groupby('position_bucket')['ctr'].mean()
print("Expected CTR by bucket (page-level):")
print(expected_ctr_by_bucket_page)

# Step 2: opportunity gap
page_level['expected_ctr'] = page_level['position_bucket'].map(expected_ctr_by_bucket_page)
page_level['opportunity_gap'] = page_level['expected_ctr'] - page_level['ctr']

# Step 3: confidence-weighted score (using total_impressions now)
page_level['confidence_weighted_score'] = page_level['opportunity_gap'] * np.log1p(page_level['total_impressions'])

# Step 4: reason code (reuse the same function — just rename impressions column to match)
def assign_reason_code_page(row):
    if row['total_impressions'] < 10:
        return 'low_confidence_signal'
    elif row['opportunity_gap'] <= 0:
        return 'on_par_or_above'
    elif row['position_bucket'] in ('top_3', 'top_10'):
        return 'high_position_low_ctr'
    elif row['position_bucket'] == 'top_20':
        return 'mid_position_low_ctr'
    else:
        return 'monitor_only'

page_level['reason_code'] = page_level.apply(assign_reason_code_page, axis=1)

# Step 5: rank
ranked_pages = page_level.sort_values('confidence_weighted_score', ascending=False).reset_index(drop=True)

print("\nTotal unique ranked pages:", len(ranked_pages))
print(ranked_pages[['content_hash_id', 'position_bucket', 'total_impressions', 'ctr',
                     'opportunity_gap', 'confidence_weighted_score', 'reason_code']].head(15))

Expected CTR by bucket (page-level):
position_bucket
beyond_20    0.001928
top_10       0.004926
top_20       0.003211
top_3        0.012399
Name: ctr, dtype: float64

Total unique ranked pages: 176738
             content_hash_id position_bucket  total_impressions       ctr  \
0   content_306bc78dff1eb683           top_3              80821  0.000433   
1   content_8d7d99f109e19aa2           top_3             203497  0.001420   
2   content_fc67675904376267           top_3              60172  0.000299   
3   content_c46df0fa61530d86           top_3              70398  0.000597   
4   content_d61fc394d10cba41           top_3              38000  0.000026   
5   content_9ef3d7516483e665           top_3              89229  0.001031   
6   content_b2b85c287474668d           top_3              65304  0.000934   
7   content_7f52754cb72a5991           top_3              43135  0.000649   
8   content_252aa5480bb1f8d7           top_3              66698  0.001124   
9   content_b9acd1ebff7d34ff

In [12]:
top_20 = ranked_pages.head(20)[[
    'content_hash_id', 'position_bucket', 'total_impressions',
    'ctr', 'expected_ctr', 'opportunity_gap', 'confidence_weighted_score', 'reason_code'
]].copy()

print(top_20.to_string())

             content_hash_id position_bucket  total_impressions       ctr  expected_ctr  opportunity_gap  confidence_weighted_score            reason_code
0   content_306bc78dff1eb683           top_3              80821  0.000433      0.012399         0.011966                   0.135220  high_position_low_ctr
1   content_8d7d99f109e19aa2           top_3             203497  0.001420      0.012399         0.010979                   0.134204  high_position_low_ctr
2   content_fc67675904376267           top_3              60172  0.000299      0.012399         0.012100                   0.133163  high_position_low_ctr
3   content_c46df0fa61530d86           top_3              70398  0.000597      0.012399         0.011803                   0.131742  high_position_low_ctr
4   content_d61fc394d10cba41           top_3              38000  0.000026      0.012399         0.012373                   0.130479  high_position_low_ctr
5   content_9ef3d7516483e665           top_3              89229  0.001

## 4. Weak picks + leakage check

## Weak picks + leakage check

### Weak pick

Row 4 (content_d61fc394d10cba41) stands out even among the top 20 — its CTR (0.003%) is effectively zero despite 38,000 impressions, meaning almost no one clicked across tens of thousands of views. While the rule correctly flags this as the largest opportunity, a CTR this extreme is more consistent with a technical problem (broken link, incorrect canonical tag, title not rendering as expected in search results) than a simple "weak copy" issue. I'm flagging this as needing a technical audit before a content rewrite, not assuming metadata alone is the fix — labeling it here as directional, not definitive.

### Leakage check

The score (opportunity_gap, confidence_weighted_score) is computed only from position_bucket, total_impressions, and the group-level expected_ctr — none of which use gsc_clicks, GA4 metrics, or any post-click behavior as a direct input. total_clicks appears in the output CSV only for transparency, so a reviewer can verify the ctr column, but it does not feed the ranking formula itself. All data is restricted to the March 2026 window — no future months were used, so there is no forward-looking leakage either.

In [13]:
# Confirm no leaked columns made it into the final scoring/output
leaked_check_cols = ['gsc_clicks', 'total_clicks', 'ga4_pageviews', 'ga4_sessions', 'scroll_events']
score_input_cols = ['position_bucket', 'total_impressions', 'expected_ctr']  # what the SCORE formula actually used

print("Columns used to compute confidence_weighted_score:", score_input_cols)
print("\nNote: total_clicks appears in the output CSV for transparency (so a reviewer can verify ctr),")
print("but it is NOT an input to opportunity_gap or confidence_weighted_score — those use expected_ctr")
print("(derived from position_bucket group averages) and total_impressions only.")

# Double check: no future-window data (April+) leaked into March scoring
print("\nDate range check — confirm we only used March 2026:")
print(df_march['report_date'].min(), "to", df_march['report_date'].max())

Columns used to compute confidence_weighted_score: ['position_bucket', 'total_impressions', 'expected_ctr']

Note: total_clicks appears in the output CSV for transparency (so a reviewer can verify ctr),
but it is NOT an input to opportunity_gap or confidence_weighted_score — those use expected_ctr
(derived from position_bucket group averages) and total_impressions only.

Date range check — confirm we only used March 2026:
2026-03-01 to 2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.